In [1]:
"""
GOLD LAYER
----------
Build a star schema from silver_sales:
  - dim_customer, dim_product, dim_location, dim_date (dimension tables)
  - fact_sales (fact table, references dimensions via surrogate keys)
"""

import duckdb

DB_PATH = "warehouse.duckdb"

def run():
    con = duckdb.connect(DB_PATH)

    # ---- dim_customer ----
    # One row per customer_id. Only attributes that describe the PERSON,
    # not where any individual order shipped to. GROUP BY forces
    # uniqueness on customer_id; MIN() picks one consistent value for
    # any column that might vary slightly across that customer's rows.
    con.execute("""
        CREATE OR REPLACE TABLE dim_customer AS
        SELECT
            customer_id,
            MIN(customer_name) AS customer_name,
            MIN(segment) AS segment
        FROM silver_sales
        GROUP BY customer_id
    """)

    # ---- dim_product ----
    # One row per product_id. Same GROUP BY + MIN() pattern to collapse
    # any slightly inconsistent product_name text under the same ID.
    con.execute("""
        CREATE OR REPLACE TABLE dim_product AS
        SELECT
            product_id,
            MIN(product_name) AS product_name,
            MIN(category) AS category,
            MIN(sub_category) AS sub_category
        FROM silver_sales
        GROUP BY product_id
    """)

    # ---- dim_location ----
    # NEW: shipping location is its own dimension, since one customer
    # ships to many different locations across orders. Surrogate key
    # generated with ROW_NUMBER() since there's no natural unique ID
    # for a location combination in the source data.
    con.execute("""
        CREATE OR REPLACE TABLE dim_location AS
        SELECT
            ROW_NUMBER() OVER () AS location_key,
            city,
            state,
            postal_code,
            region,
            country
        FROM (
            SELECT DISTINCT city, state, postal_code, region, country
            FROM silver_sales
        )
    """)

    # ---- dim_date ----
    con.execute("""
        CREATE OR REPLACE TABLE dim_date AS
        WITH all_dates AS (
            SELECT DISTINCT order_date AS the_date FROM silver_sales
            UNION
            SELECT DISTINCT ship_date AS the_date FROM silver_sales
        )
        SELECT
            CAST(STRFTIME(the_date, '%Y%m%d') AS INTEGER) AS date_key,
            the_date AS full_date,
            EXTRACT(YEAR FROM the_date) AS year,
            EXTRACT(MONTH FROM the_date) AS month,
            STRFTIME(the_date, '%B') AS month_name,
            EXTRACT(QUARTER FROM the_date) AS quarter,
            EXTRACT(DAY FROM the_date) AS day,
            STRFTIME(the_date, '%A') AS day_name,
            EXTRACT(DOW FROM the_date) AS day_of_week
        FROM all_dates
        ORDER BY full_date
    """)

    # ---- fact_sales ----
    # Joins to dim_location to look up the surrogate location_key for
    # each transaction's actual city/state/postal_code/region/country.
    con.execute("""
        CREATE OR REPLACE TABLE fact_sales AS
        SELECT
            s.row_id AS sale_id,
            s.order_id,
            s.customer_id,
            s.product_id,
            l.location_key,
            CAST(STRFTIME(s.order_date, '%Y%m%d') AS INTEGER) AS order_date_key,
            CAST(STRFTIME(s.ship_date, '%Y%m%d') AS INTEGER) AS ship_date_key,
            s.ship_mode,
            s.sales AS revenue,
            s.quantity,
            s.discount,
            s.profit
        FROM silver_sales s
        JOIN dim_location l
            ON s.city = l.city
            AND s.state = l.state
            AND s.postal_code = l.postal_code
            AND s.region = l.region
            AND s.country = l.country
    """)

    # ---- sanity checks ----
    for table in ["dim_customer", "dim_product", "dim_location", "dim_date", "fact_sales"]:
        count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table}: {count} rows")

    # confirm each dimension key is truly unique now
    for table, key in [("dim_customer", "customer_id"), ("dim_product", "product_id"), ("dim_location", "location_key")]:
        total = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        unique = con.execute(f"SELECT COUNT(DISTINCT {key}) FROM {table}").fetchone()[0]
        status = "OK" if total == unique else "MISMATCH"
        print(f"{table}: {total} rows, {unique} unique {key} -> {status}")

    con.close()

if __name__ == "__main__":
    run()

dim_customer: 793 rows
dim_product: 1862 rows
dim_location: 632 rows
dim_date: 1434 rows
fact_sales: 9994 rows
dim_customer: 793 rows, 793 unique customer_id -> OK
dim_product: 1862 rows, 1862 unique product_id -> OK
dim_location: 632 rows, 632 unique location_key -> OK


In [2]:
import duckdb
con = duckdb.connect("warehouse.duckdb")
result = con.execute("""
    SELECT
        p.category,
        d.year,
        d.month_name,
        SUM(f.revenue) AS total_revenue
    FROM fact_sales f
    JOIN dim_product p ON f.product_id = p.product_id
    JOIN dim_date d ON f.order_date_key = d.date_key
    GROUP BY p.category, d.year, d.month_name
    ORDER BY d.year, total_revenue DESC
    LIMIT 10
""").fetchdf()
print(result.to_string())
con.close()

          category  year month_name  total_revenue
0       Technology  2014      March     32511.1740
1        Furniture  2014   December     30645.9665
2       Technology  2014  September     30537.5720
3       Technology  2014   November     30201.4080
4  Office Supplies  2014  September     27423.2980
5  Office Supplies  2014   November     26862.4360
6        Furniture  2014  September     23816.4808
7        Furniture  2014   November     21564.8727
8       Technology  2014   December     20893.1920
9  Office Supplies  2014   December     18006.4620


In [3]:
import duckdb
con = duckdb.connect("warehouse.duckdb")

result = con.execute("""
    WITH monthly_sales AS (
        SELECT
            d.year,
            d.month,
            d.month_name,
            p.category,
            SUM(f.revenue) AS total_revenue
        FROM fact_sales f
        JOIN dim_date d ON f.order_date_key = d.date_key
        JOIN dim_product p ON f.product_id = p.product_id
        GROUP BY d.year, d.month, d.month_name, p.category
    )
    SELECT
        *,
        LAG(total_revenue) OVER (PARTITION BY category ORDER BY year, month) AS prev_month_revenue,
        ROUND(
            (total_revenue - LAG(total_revenue) OVER (PARTITION BY category ORDER BY year, month))
            / LAG(total_revenue) OVER (PARTITION BY category ORDER BY year, month) * 100,
            2
        ) AS mom_growth_pct
    FROM monthly_sales
    ORDER BY category, year, month
""").fetchdf()

print(result.to_string())
con.close()

     year  month month_name         category  total_revenue  prev_month_revenue  mom_growth_pct
0    2014      1    January        Furniture      6242.5250                 NaN             NaN
1    2014      2   February        Furniture      1839.6580           6242.5250          -70.53
2    2014      3      March        Furniture     14573.9560           1839.6580          692.21
3    2014      4      April        Furniture      7944.8370          14573.9560          -45.49
4    2014      5        May        Furniture      6912.7870           7944.8370          -12.99
5    2014      6       June        Furniture     13206.1256           6912.7870           91.04
6    2014      7       July        Furniture     10821.0510          13206.1256          -18.06
7    2014      8     August        Furniture      7320.3465          10821.0510          -32.35
8    2014      9  September        Furniture     23816.4808           7320.3465          225.35
9    2014     10    October        Furni

In [4]:
"""
GOLD LAYER
----------
Build a star schema from silver_sales:
  dim_customer, dim_product, dim_location, dim_date  (dimensions)
  fact_sales                                          (fact table)

Note on dim_customer / dim_product:
GROUP BY + MIN() is used instead of SELECT DISTINCT. The raw data has
the same customer_id shipping to many different locations, and some
product_ids with slightly inconsistent product_name text. SELECT DISTINCT
would have produced duplicate rows per key (a real bug caught during
development - see README). GROUP BY forces one row per key; MIN() picks
one deterministic value for any column that varies slightly within a group.
"""

import duckdb

DB_PATH = "warehouse.duckdb"


def run():
    con = duckdb.connect(DB_PATH)

    # ---- dim_customer: one row per customer_id ----
    con.execute("""
        CREATE OR REPLACE TABLE dim_customer AS
        SELECT
            customer_id,
            MIN(customer_name) AS customer_name,
            MIN(segment) AS segment
        FROM silver_sales
        GROUP BY customer_id
    """)

    # ---- dim_product: one row per product_id ----
    con.execute("""
        CREATE OR REPLACE TABLE dim_product AS
        SELECT
            product_id,
            MIN(product_name) AS product_name,
            MIN(category) AS category,
            MIN(sub_category) AS sub_category
        FROM silver_sales
        GROUP BY product_id
    """)

    # ---- dim_location: one row per unique ship-to location ----
    # Surrogate key via ROW_NUMBER() since there's no natural unique ID
    # for a location combination in the source data.
    con.execute("""
        CREATE OR REPLACE TABLE dim_location AS
        SELECT
            ROW_NUMBER() OVER () AS location_key,
            city,
            state,
            postal_code,
            region,
            country
        FROM (
            SELECT DISTINCT city, state, postal_code, region, country
            FROM silver_sales
        )
    """)

    # ---- dim_date: one row per calendar date present in the data ----
    con.execute("""
        CREATE OR REPLACE TABLE dim_date AS
        WITH all_dates AS (
            SELECT DISTINCT order_date AS the_date FROM silver_sales
            UNION
            SELECT DISTINCT ship_date AS the_date FROM silver_sales
        )
        SELECT
            CAST(STRFTIME(the_date, '%Y%m%d') AS INTEGER) AS date_key,
            the_date AS full_date,
            EXTRACT(YEAR FROM the_date) AS year,
            EXTRACT(MONTH FROM the_date) AS month,
            STRFTIME(the_date, '%B') AS month_name,
            EXTRACT(QUARTER FROM the_date) AS quarter,
            EXTRACT(DAY FROM the_date) AS day,
            STRFTIME(the_date, '%A') AS day_name,
            EXTRACT(DOW FROM the_date) AS day_of_week
        FROM all_dates
        ORDER BY full_date
    """)

    # ---- fact_sales: one row per transaction line, keys + measures only ----
    con.execute("""
        CREATE OR REPLACE TABLE fact_sales AS
        SELECT
            s.row_id AS sale_id,
            s.order_id,
            s.customer_id,
            s.product_id,
            l.location_key,
            CAST(STRFTIME(s.order_date, '%Y%m%d') AS INTEGER) AS order_date_key,
            CAST(STRFTIME(s.ship_date, '%Y%m%d') AS INTEGER) AS ship_date_key,
            s.ship_mode,
            s.sales AS revenue,
            s.quantity,
            s.discount,
            s.profit
        FROM silver_sales s
        JOIN dim_location l
            ON s.city = l.city
            AND s.state = l.state
            AND s.postal_code = l.postal_code
            AND s.region = l.region
            AND s.country = l.country
    """)

    # ---- sanity checks: row counts + dimension key uniqueness ----
    print("\nRow counts:")
    for table in ["dim_customer", "dim_product", "dim_location", "dim_date", "fact_sales"]:
        count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"  {table}: {count} rows")

    print("\nDimension key uniqueness check:")
    for table, key in [
        ("dim_customer", "customer_id"),
        ("dim_product", "product_id"),
        ("dim_location", "location_key"),
    ]:
        total = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        unique = con.execute(f"SELECT COUNT(DISTINCT {key}) FROM {table}").fetchone()[0]
        status = "OK" if total == unique else "MISMATCH"
        print(f"  {table}: {total} rows, {unique} unique {key} -> {status}")

    con.close()


if __name__ == "__main__":
    run()


Row counts:
  dim_customer: 793 rows
  dim_product: 1862 rows
  dim_location: 632 rows
  dim_date: 1434 rows
  fact_sales: 9994 rows

Dimension key uniqueness check:
  dim_customer: 793 rows, 793 unique customer_id -> OK
  dim_product: 1862 rows, 1862 unique product_id -> OK
  dim_location: 632 rows, 632 unique location_key -> OK
